In [ ]:
# Perspective-matched car floor reflection: Google Colab + Gradio
# Fully automatic. Paste this whole file into one Colab cell and run it.
#
# How it works: a floor reflection mirrors every part of the car about the floor directly
# beneath it, and that floor is different for different parts:
#   - the side of the body stands over the line through the same-side tyre contacts
#   - the front/rear of the body stands over a line under the bumper, found from the
#     bumper's own bottom edge (so a hidden far-side tyre is never needed)
#   - each tyre is mirrored about its own contact with the floor

import subprocess, sys
from importlib.metadata import version
# Keep Colab's own NumPy: letting pip replace it under a running session breaks imports.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio", "rembg", "onnxruntime", "ultralytics",
                "numpy==" + version("numpy")], check=True)

import cv2
import numpy as np
import gradio as gr
from rembg import remove, new_session
from ultralytics import YOLO

SESSION = new_session("isnet-general-use")
DET = YOLO("yolov8m-oiv7.pt")                      # Open Images model: has Wheel and Tire classes
WHEEL_IDS = [k for k, v in DET.names.items() if v in ("Wheel", "Tire")]


def to_rgba(img):
    """Use the upload's own alpha if it has real transparency, otherwise cut the car out."""
    if img.shape[2] == 4 and img[..., 3].min() < 250:
        return img
    return np.array(remove(np.ascontiguousarray(img[..., :3]), session=SESSION))


def detect_wheels(car):
    """Wheel/tyre boxes, detected on the car over a plain background, cropped to the car."""
    ys, xs = np.where(car[..., 3] > 127)
    if len(xs) == 0 or not WHEEL_IDS:
        return []
    a = car[..., 3:4].astype(np.float32) / 255
    rgb = (car[..., :3] * a + 190 * (1 - a)).astype(np.uint8)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    m = int(0.06 * max(x1 - x0, y1 - y0))
    X0, Y0 = max(0, x0 - m), max(0, y0 - m)
    X1, Y1 = min(car.shape[1], x1 + m), min(car.shape[0], y1 + m)
    crop = np.ascontiguousarray(rgb[Y0:Y1, X0:X1, ::-1])
    res = DET.predict(crop, classes=WHEEL_IDS, conf=0.08, imgsz=1024, agnostic_nms=True, verbose=False)[0]
    return [[b[0] + X0, b[1] + Y0, b[2] + X0, b[3] + Y0] for b in res.boxes.xyxy.cpu().numpy().tolist()]

# ---------- geometry of the floor under the car ----------
# A floor reflection mirrors every car pixel about the floor point directly beneath it.
# The floor under a car is not one line:
#   * the side of the body stands over the line through the same-side tyre contacts
#   * the front/rear of the body stands over a line UNDER THE BUMPER, which is ahead of
#     the axle by the overhang. It is found from the bumper's own bottom edge.
#   * every tyre touches the floor at its own contact patch
# So the body and each wheel are mirrored separately, each about its own floor line.

FACE_ON_ASPECT = 0.70      # wheel width/height above this = mostly the wheel's face is showing
SIDE_ON_ASPECT = 0.90      # wheel width/height above this = car seen from the side
FULL_ARCH_DEPTH = 0.40     # wheel rises this far (of its height) above the body's bottom edge
BUMPER_VS_SILL = 1.10      # bumper bottom sits a little higher off the floor than the sill
DEFAULT_CLEARANCE = 0.28   # body-to-floor gap as a fraction of wheel height
END_GAP = 0.12             # floor sits this far (of wheel height) below the bumper when no tyre shows under it
END_ON_CLEARANCE = 0.12    # body-to-floor gap as a fraction of car width (front/rear views)


def lower_envelope(mask):
    """Lowest car pixel in each column (NaN where the column is empty)."""
    h = mask.shape[0]
    any_col = mask.any(axis=0)
    yb = (h - 1 - np.argmax(mask[::-1], axis=0)).astype(np.float32)
    yb[~any_col] = np.nan
    return yb


def merge_boxes(boxes):
    """Wheel and Tire detections of the same wheel overlap: merge them into one box."""
    boxes = [list(map(float, b)) for b in boxes]
    changed = True
    while changed:
        changed = False
        out = []
        for b in boxes:
            for o in out:
                ix = max(0.0, min(b[2], o[2]) - max(b[0], o[0]))
                iy = max(0.0, min(b[3], o[3]) - max(b[1], o[1]))
                inter = ix * iy
                small = min((b[2] - b[0]) * (b[3] - b[1]), (o[2] - o[0]) * (o[3] - o[1]))
                if small > 0 and inter / small > 0.5:
                    o[0], o[1] = min(o[0], b[0]), min(o[1], b[1])
                    o[2], o[3] = max(o[2], b[2]), max(o[3], b[3])
                    changed = True
                    break
            else:
                out.append(b)
        boxes = out
    return boxes


def robust_line(x, y, tol, iters=400, seed=0):
    """RANSAC straight line through the dominant run of points. Returns slope, intercept."""
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)
    n = len(x)
    if n < 2:
        return None
    if n < 6:
        s, i = np.polyfit(x, y, 1)
        return float(s), float(i)
    rng = np.random.default_rng(seed)
    span = x.max() - x.min()
    best, best_n = None, -1
    for _ in range(iters):
        a, b = rng.integers(0, n, 2)
        if abs(x[a] - x[b]) < 0.15 * span:
            continue
        s = (y[b] - y[a]) / (x[b] - x[a])
        i = y[a] - s * x[a]
        inl = np.abs(y - (s * x + i)) < tol
        k = int(inl.sum())
        if k > best_n:
            best_n, best = k, inl
    if best is None or best_n < 2:
        s, i = np.polyfit(x, y, 1)
        return float(s), float(i)
    s, i = np.polyfit(x[best], y[best], 1)
    return float(s), float(i)


def analyse(alpha, boxes):
    """Work out the floor lines. Returns a dict describing the layout."""
    mask = alpha > 127
    h, w = mask.shape
    cols = np.where(mask.any(axis=0))[0]
    rows = np.where(mask.any(axis=1))[0]
    out = {"ok": False, "wheels": [], "mode": "none"}
    if len(cols) == 0:
        return out
    cx0, cx1 = int(cols[0]), int(cols[-1])
    cy0, cy1 = int(rows[0]), int(rows[-1])
    car_w, car_h = cx1 - cx0 + 1, cy1 - cy0 + 1
    ybot = lower_envelope(mask)
    xs_all = np.arange(w)
    out.update(car_box=[cx0, cy0, cx1, cy1], car_w=car_w, car_h=car_h)

    # ---- keep only detections that are real wheels standing on the floor ----
    wheels = []
    for b in merge_boxes(boxes):
        x1, y1, x2, y2 = b
        xa, xb = int(max(0, np.floor(x1))), int(min(w - 1, np.ceil(x2)))
        if xb - xa < 3:
            continue
        seg = ybot[xa:xb + 1]
        if np.all(np.isnan(seg)):
            continue
        low = float(np.nanmax(seg))                      # lowest car pixel under this box
        bh = y2 - y1
        if bh < 0.04 * car_h:
            continue
        if low > y2 + 0.25 * bh or low < y2 - 0.25 * bh:  # box bottom is not the car's bottom there
            continue
        if y2 < cy0 + 0.5 * car_h:
            continue
        wheels.append({"box": [x1, y1, x2, y2], "low": low})

    def in_wheel(xq, pad=0.06, skip=None):
        m = np.zeros(len(xq), bool)
        for d in wheels:
            if d is skip:
                continue
            x1, _, x2, _ = d["box"]
            p = pad * (x2 - x1)
            m |= (xq >= x1 - p) & (xq <= x2 + p)
        return m

    # ---- wheel with its face visible (sits up inside an arch) or tyre peeking under a bumper ----
    for d in wheels:
        x1, y1, x2, y2 = d["box"]
        bw, bh = x2 - x1, y2 - y1
        be = max(bw, 0.6 * bh)
        vals = []
        for a, b in ((x1 - 0.50 * be, x1 - 0.08 * be), (x2 + 0.08 * be, x2 + 0.50 * be)):
            xq = xs_all[(xs_all >= a) & (xs_all <= b)]
            xq = xq[~in_wheel(xq, skip=d)]
            v = ybot[xq]
            v = v[~np.isnan(v)]
            if len(v) >= 3:
                vals.append(float(np.median(v)))
        d["arch"] = (max(vals) - y1) / bh if vals else 0.0
        d["full"] = d["arch"] > FULL_ARCH_DEPTH
        d["aspect"] = bw / max(bh, 1.0)

    out["wheels"] = wheels
    # A wheel whose face is visible is a tall box; a tyre peeking under a bumper is a wide, short one.
    tall = [d for d in wheels if d["aspect"] < 1.4]
    hmax = max([d["box"][3] - d["box"][1] for d in tall], default=0.0)
    for d in wheels:
        d["full"] = d["aspect"] < 1.4 and (d["box"][3] - d["box"][1]) >= 0.5 * hmax
    full = sorted([d for d in wheels if d["full"]], key=lambda d: d["box"][3] - d["box"][1], reverse=True)[:2]
    if len(full) == 2:
        l, r = sorted(full, key=lambda d: d["box"][0])
        # both tyres at the two extremes of the car = looking straight at the front or rear
        if l["box"][0] - cx0 < 0.08 * car_w and cx1 - r["box"][2] < 0.08 * car_w:
            full = []
    for d in wheels:
        d["full"] = any(d is f for f in full)
    tol = max(2.0, 0.012 * car_h)

    def env_fit(xa, xb, trim_outer=None):
        """Straight line along the body's bottom edge between columns xa..xb (wheels skipped)."""
        xa, xb = int(max(cx0, xa)), int(min(cx1, xb))
        if xb - xa < 0.04 * car_w:
            return None
        xq = np.arange(xa, xb + 1)
        keep = ~in_wheel(xq, pad=0.10) & ~np.isnan(ybot[xq])
        if trim_outer == "left":
            keep &= xq > xa + 0.08 * (xb - xa)
        elif trim_outer == "right":
            keep &= xq < xb - 0.08 * (xb - xa)
        elif trim_outer == "both":
            keep &= (xq > xa + 0.06 * (xb - xa)) & (xq < xb - 0.06 * (xb - xa))
        if keep.sum() < 8:
            return None
        return robust_line(xq[keep], ybot[xq[keep]], tol)

    def contact(d, slope, side):
        """Where the tyre meets the floor: the tyre's bottom edge, touched by a line of the
        given slope. Returns the end of that contact strip on the requested side."""
        x1, y1, x2, y2 = d["box"]
        xq = np.arange(int(max(0, x1)), int(min(w - 1, x2)) + 1)
        v = ybot[xq]
        ok = ~np.isnan(v) & ~(v > y2 + 0.08 * (y2 - y1))     # ignore anything hanging below the tyre
        if not ok.any():
            ok = ~np.isnan(v)
        xq, v = xq[ok], v[ok]
        r = v - slope * xq
        m = r.max()
        t = max(1.5, 0.02 * (y2 - y1))
        on = xq[r >= m - t]
        px = float(on.max() if side > 0 else on.min())
        return [px, float(m + slope * px)], [float(on.min()), float(on.max())]

    def tyre_edge(d):
        """The straight line the tyre itself rests on: the longest flat edge along the
        tyre's bottom outline. A round bottom (wheel seen from the side) gives a level line."""
        x1, y1, x2, y2 = d["box"]
        bw, bh = x2 - x1, y2 - y1
        xq = np.arange(int(max(0, x1)), int(min(w - 1, x2)) + 1)
        v = ybot[xq]
        ok = ~np.isnan(v) & ~(v > y2 + 0.08 * bh)
        if ok.sum() < 3:
            return (0.0, float(0.5 * (x1 + x2)), float(y2))
        xq, v = xq[ok], v[ok]
        k = int(np.argmax(v))
        best = (0.0, float(xq[k]), float(v[k]))
        hull = cv2.convexHull(np.stack([xq, v], 1).astype(np.float32))[:, 0, :]
        span = 0.25 * bw
        for p, q in zip(hull, np.roll(hull, -1, axis=0)):
            dx = float(q[0] - p[0])
            if abs(dx) < span or min(p[1], q[1]) < v.max() - 0.2 * bh:
                continue
            sl = float(q[1] - p[1]) / dx
            mid = 0.5 * (p + q)
            if abs(sl) <= 0.45 and mid[1] > np.interp(mid[0], xq, v) - 1.0:   # an edge on the underside
                span = abs(dx)
                best = (sl, float(mid[0]), float(mid[1]))
        return best

    lines = {}

    if len(full) >= 2:
        l, r = sorted(full, key=lambda d: d["box"][0])
        # the corner wheel is the one with the car's front/rear face stretching away beyond it
        corner, other = (l, r) if (l["box"][0] - cx0) >= (cx1 - r["box"][2]) else (r, l)
        side = 1 if corner is l else -1
        side_on = 0.5 * (corner["aspect"] + other["aspect"]) > SIDE_ON_ASPECT
        end = None
        if not side_on:
            if side > 0:
                end = env_fit(cx0, corner["box"][0], "left")
            else:
                end = env_fit(corner["box"][2], cx1, "right")
        s_e = end[0] if end else 0.0
        for d in wheels:
            d["pt"], d["strip"] = contact(d, s_e, side)
        (xa, ya), (xb, yb) = corner["pt"], other["pt"]
        s_s = (yb - ya) / (xb - xa) if abs(xb - xa) > 1 else 0.0
        # The side floor line is the line that touches both tyres from below.
        for _ in range(3):
            pa, _ = contact(corner, s_s, side)
            pb, _ = contact(other, s_s, side)
            if abs(pb[0] - pa[0]) > 1:
                s_s = (pb[1] - pa[1]) / (pb[0] - pa[0])
        corner["side_pt"], other["side_pt"] = pa, pb
        i_s = pa[1] - s_s * pa[0]
        lines["side"] = (s_s, i_s)
        # body-to-floor gap, measured on the sill next to the corner wheel
        if side > 0:
            a, b = corner["box"][2], other["box"][0]
            a, b = a + 0.05 * (b - a), a + 0.45 * (b - a)
        else:
            a, b = other["box"][2], corner["box"][0]
            a, b = b - 0.45 * (b - a), b - 0.05 * (b - a)
        xq = np.arange(int(a), int(b) + 1)
        xq = xq[(xq >= 0) & (xq < w)]
        gap = (s_s * xq + i_s) - ybot[xq]
        gap = gap[~np.isnan(gap)]
        bh = corner["box"][3] - corner["box"][1]
        c = float(np.percentile(gap, 60)) if len(gap) >= 5 else DEFAULT_CLEARANCE * bh
        c = float(np.clip(c, 0.18 * bh, 0.34 * bh))
        if end:
            lines["end"] = (end[0], end[1] + BUMPER_VS_SILL * c)
            lines["end_edge"] = end
        out.update(mode="side" if side_on else "angled", side=side, clearance=c, corner=corner, other=other)
    elif len(full) == 1:
        corner = full[0]
        bh = corner["box"][3] - corner["box"][1]
        c = DEFAULT_CLEARANCE * bh
        left = env_fit(cx0, corner["box"][0], "left")
        right = env_fit(corner["box"][2], cx1, "right")
        wl = corner["box"][0] - cx0
        wr = cx1 - corner["box"][2]
        end, sidefit = (left, right) if wl >= wr else (right, left)
        side = 1 if wl >= wr else -1
        s_e = end[0] if end else 0.0
        for d in wheels:
            d["pt"], d["strip"] = contact(d, s_e, side)
        if sidefit:
            s_s = sidefit[0]
            lines["side"] = (s_s, corner["pt"][1] - s_s * corner["pt"][0])
        if end:
            lines["end"] = (end[0], end[1] + BUMPER_VS_SILL * c)
            lines["end_edge"] = end
        out.update(mode="angled", side=side, clearance=c, corner=corner)
    else:
        # front or rear view: only tyre treads show under the bumper
        c = END_ON_CLEARANCE * car_w
        end = env_fit(cx0, cx1, "both")
        s_e = end[0] if end else 0.0
        for d in wheels:
            d["pt"], d["strip"] = contact(d, s_e, 1)
        if end:
            lines["end"] = (end[0], end[1] + c)
            lines["end_edge"] = end
        elif wheels:
            lines["end"] = (0.0, max(d["pt"][1] for d in wheels))
        else:
            lines["end"] = (0.0, float(cy1))
        out.update(mode="end", side=0, clearance=c)

    if not lines:
        lines["end"] = (0.0, float(cy1))
    out["lines"] = lines
    out["axle_slope"] = lines["end_edge"][0] if "end_edge" in lines else 0.0
    for d in wheels:
        d["edge"] = tyre_edge(d)

    # ---- tyres the detector missed: anything hanging well below the body's bottom edge ----
    c = out["clearance"]
    x = np.arange(w, dtype=np.float32)
    refs = []
    if "side" in lines:
        refs.append(lines["side"][0] * x + lines["side"][1] - c)
    if "end_edge" in lines:
        refs.append(lines["end_edge"][0] * x + lines["end_edge"][1])
    ref = np.min(refs, axis=0) if refs else np.full(w, cy1 - c, np.float32)   # the body's bottom edge
    out["body_edge"] = ref
    yy = np.arange(h)[:, None]
    hang = mask & (yy > (ref + 0.3 * c)[None, :])
    for d in wheels:
        x1, y1, x2, y2 = d["box"]
        p = 0.1 * (x2 - x1)
        hang[:, int(max(0, x1 - p)):int(min(w, x2 + p)) + 1] = False
    n, lab, stats, _ = cv2.connectedComponentsWithStats(hang.astype(np.uint8), 8)
    for k in range(1, n):
        bx, by, bw_, bh_, area = stats[k]
        if bh_ < 0.03 * car_h or bw_ < 0.02 * car_w or area < 0.3 * bw_ * bh_:
            continue
        d = {"box": [float(bx), float(by), float(bx + bw_), float(by + bh_)], "full": False,
             "aspect": bw_ / max(bh_, 1), "arch": 0.0, "blob": True}
        d["pt"], d["strip"] = contact(d, out["axle_slope"], 1)
        d["edge"] = tyre_edge(d)
        wheels.append(d)

    # ---- floor line under the front/rear face ----
    # If a tyre shows under the bumper, the floor there is where that tyre stands.
    # Otherwise it sits a little below the bumper's bottom edge.
    if "end_edge" in lines:
        s_e, i_e = lines["end_edge"]
        ref_h = (out["corner"]["box"][3] - out["corner"]["box"][1]) if "corner" in out else 0.3 * car_w
        peeks = [d for d in wheels if not d["full"]]
        if out["mode"] == "angled" and "corner" in out:
            cxc = 0.5 * (out["corner"]["box"][0] + out["corner"]["box"][2])
            peeks = [d for d in peeks if (0.5 * (d["box"][0] + d["box"][2]) - cxc) * out["side"] < 0]
        # lowest thing under the front/rear face: a tyre showing under the bumper (detected
        # or not), or the bumper's own bottom edge when no tyre shows
        if out["mode"] == "angled" and "corner" in out:
            cb = out["corner"]["box"]
            pad = 0.1 * (cb[2] - cb[0])
            xa, xb = (cx0, cb[0] - pad) if out["side"] > 0 else (cb[2] + pad, cx1)
        else:
            xa, xb = cx0, cx1
        xq = np.arange(int(max(0, xa)), int(min(w - 1, xb)) + 1)
        v = ybot[xq] - s_e * xq
        cand = [d["pt"][1] - s_e * d["pt"][0] for d in peeks]
        if len(xq) and not np.all(np.isnan(v)):
            cand.append(float(np.nanmax(v)))
        i_new = max(cand) if cand else i_e
        if not peeks:
            # Nothing but the bumper itself is down there. A bumper hangs above the floor, so its
            # reflection must start a little below it; touching, it would look like the bumper
            # had grown downwards.
            i_new += END_GAP * ref_h
        i_new = float(np.clip(i_new, i_e, i_e + 0.40 * ref_h))
        lines["end"] = (s_e, i_new)
    out["ok"] = True
    return out


def body_floor(model, w):
    """Floor line under the body for every column."""
    x = np.arange(w, dtype=np.float32)
    L = [s * x + i for k, (s, i) in model["lines"].items() if k in ("side", "end")]
    return np.min(L, axis=0).astype(np.float32)


def floor_line(model, w):
    """ONE continuous floor line for the whole car, so the reflection is one unbroken image.
    Under each wheel it is the straight line that tyre itself rests on, so the wheel is mirrored
    directly beneath itself, lined up with the real tyre; between wheels it eases smoothly into
    the floor line under the body."""
    x = np.arange(w, dtype=np.float32)
    B = body_floor(model, w)
    num = np.zeros(w, np.float32)
    den = np.zeros(w, np.float32)
    for d in model["wheels"]:
        x1, y1, x2, y2 = d["box"]
        bw, bh = x2 - x1, y2 - y1
        lift = 0.015 * bh                                  # tuck under the tyre so no gap shows
        sl, ex, ey = d["edge"]                             # the line the tyre itself rests on
        line = ey - lift + sl * (x - ex)
        m = max(8.0, 0.3 * bw)                             # ease in/out over this many columns
        t = np.clip(np.minimum(x - (x1 - m), (x2 + m) - x) / m, 0, 1)
        wgt = t * t * (3 - 2 * t)
        num += wgt * line
        den += wgt
    k = np.clip(den, 0, 1)
    wl = np.where(den > 0, num / np.maximum(den, 1e-6), B)
    return (B * (1 - k) + wl * k).astype(np.float32)


def wheels_only(car, model):
    """The car image with everything except its wheels made transparent, so that only the
    tyres and rims get mirrored. Each wheel is a soft-edged rounded shape around its box."""
    h, w = car.shape[:2]
    keep = np.zeros((h, w), np.float32)
    rows = np.arange(h)[:, None]
    for d in model["wheels"]:
        x1, y1, x2, y2 = d["box"]
        bw, bh = x2 - x1, y2 - y1
        m = np.zeros((h, w), np.uint8)
        xa, xb = int(max(0, x1)), int(min(w - 1, x2))
        if d["full"]:
            r = int(0.38 * min(bw, bh))                    # rounded corners leave the wheel arch out
            ya, yb = int(max(0, y1)), int(min(h - 1, y2 + 0.04 * bh))
            cv2.rectangle(m, (xa + r, ya), (xb - r, yb), 255, -1)
            cv2.rectangle(m, (xa, ya + r), (xb, yb - r), 255, -1)
            for cx, cy in ((xa + r, ya + r), (xb - r, ya + r), (xa + r, yb - r), (xb - r, yb - r)):
                cv2.circle(m, (cx, cy), r, 255, -1)
        else:
            # a tyre showing under a bumper: only the part hanging below the bumper's edge
            top = np.maximum(model["body_edge"][xa:xb + 1], max(0, y1))
            m[:, xa:xb + 1][rows >= top[None, :]] = 255
        f = max(1.0, 0.03 * bh)
        keep = np.maximum(keep, cv2.GaussianBlur(m.astype(np.float32) / 255, (0, 0), f))
    out = car.copy()
    out[..., 3] = (car[..., 3].astype(np.float32) * np.clip(keep, 0, 1)).astype(np.uint8)
    return out


def make_reflection(car, model, opacity, fade, blur):
    """Mirror the whole car about the floor line: y' = 2*B(x) - y. Returns premultiplied rgb, alpha."""
    h, w = car.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    fade_px = max(1.0, fade * model["car_h"])
    B = floor_line(model, w)
    refl = cv2.remap(car, xx, 2 * B[None, :] - yy, cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_CONSTANT, borderValue=0).astype(np.float32)
    below = yy - B[None, :]
    dist = np.clip(below / fade_px, 0, 1)
    a = refl[..., 3:4] / 255.0 * ((1 - dist) ** 1.5 * (below >= 0))[..., None]
    rgb = refl[..., :3] * a
    if blur > 0:
        # a floor reflection is crisp where it meets the car and softer further away
        near = max(0.6, 0.25 * blur)
        r0, a0 = cv2.GaussianBlur(rgb, (0, 0), near), cv2.GaussianBlur(a, (0, 0), near)[..., None]
        r1, a1 = cv2.GaussianBlur(rgb, (0, 0), blur), cv2.GaussianBlur(a, (0, 0), blur)[..., None]
        r2, a2 = cv2.GaussianBlur(rgb, (0, 0), 2.5 * blur), cv2.GaussianBlur(a, (0, 0), 2.5 * blur)[..., None]
        t = dist[..., None] * 2.0
        w1, w2 = np.clip(t, 0, 1), np.clip(t - 1, 0, 1)
        rgb = (r0 * (1 - w1) + r1 * w1) * (1 - w2) + r2 * w2
        a = (a0 * (1 - w1) + a1 * w1) * (1 - w2) + a2 * w2
    return rgb * opacity, a * opacity


def floor_texture(bg):
    """Fine detail of the floor (tile joints, rings), so the reflection sits in the floor, not on it."""
    g = cv2.cvtColor(bg, cv2.COLOR_RGB2GRAY).astype(np.float32) + 1.0
    return np.clip(g / cv2.GaussianBlur(g, (0, 0), 12), 0.55, 1.0)[..., None]


BLEND_MODES = ["Normal", "Screen", "Soft light", "Overlay", "Hard light", "Lighten", "Multiply"]


def blend_pixels(base, top, mode):
    """Photoshop-style blend of the reflection (top) into the floor (base). Both are 0..1."""
    if mode == "Screen":
        return 1 - (1 - base) * (1 - top)
    if mode == "Multiply":
        return base * top
    if mode == "Lighten":
        return np.maximum(base, top)
    if mode == "Overlay":
        return np.where(base < 0.5, 2 * base * top, 1 - 2 * (1 - base) * (1 - top))
    if mode == "Hard light":
        return np.where(top < 0.5, 2 * base * top, 1 - 2 * (1 - base) * (1 - top))
    if mode == "Soft light":
        return (1 - 2 * top) * base * base + 2 * top * base
    return top


def composite(bg, refl_rgb, refl_a, car, mode="Normal", shine=0.0):
    tex = floor_texture(bg)
    refl_rgb, refl_a = refl_rgb * tex, refl_a * tex
    base = bg.astype(np.float32) / 255.0
    top = np.clip(refl_rgb / np.maximum(refl_a, 1e-4) / 255.0, 0, 1)      # the reflection's own colour
    out = base * (1 - refl_a) + blend_pixels(base, top, mode) * refl_a
    if shine > 0:
        # glossy floor: the bright parts of the reflection (paint highlights, lamps, rims) glow on top
        lum = top.max(axis=2, keepdims=True)
        glow = top * np.clip((lum - 0.45) / 0.55, 0, 1) ** 1.5 * refl_a * shine
        out = 1 - (1 - out) * (1 - np.clip(glow, 0, 1))
    out = out * 255.0
    ca = car[..., 3:4].astype(np.float32) / 255.0
    out = out * (1 - ca) + car[..., :3].astype(np.float32) * ca
    return np.clip(out, 0, 255).astype(np.uint8)


def draw_debug(base, model):
    img = base.copy()
    h, w = img.shape[:2]
    t = max(2, w // 500)
    if not model.get("ok"):
        return img
    x0, _, x1, _ = model["car_box"]
    xs = np.arange(x0, x1 + 1)
    B = floor_line(model, w)
    cv2.polylines(img, [np.stack([xs, B[xs]], 1).astype(np.int32)], False, (0, 255, 0), t)
    for d in model["wheels"]:
        c = (int(round(d["pt"][0])), int(round(d["pt"][1])))
        cv2.circle(img, c, t * 3, (255, 0, 0), -1)
        cv2.circle(img, c, t * 3, (255, 255, 255), 1)
    return img


# ---------- UI ----------

def strip_shadow(car):
    """The car without any shadow that was already added under it.
    The picture is cut out again over white, where a shadow is only a grey smudge on the
    floor, and whatever that second cut-out drops is removed from the car's transparency."""
    a = car[..., 3:4].astype(np.float32) / 255
    flat = (car[..., :3] * a + 255 * (1 - a)).astype(np.uint8)
    keep = np.array(remove(np.ascontiguousarray(flat), session=SESSION))[..., 3]
    out = car.copy()
    out[..., 3] = np.minimum(car[..., 3], keep)
    return out


def on_upload(img, has_shadow):
    if img is None:
        return None, None, None, None
    full = to_rgba(img)                                  # what gets drawn: car plus its shadow
    car = strip_shadow(full) if has_shadow else full     # what gets measured and mirrored: car only
    model = analyse(car[..., 3], detect_wheels(car))
    a = full[..., 3:4].astype(np.float32) / 255
    base = (full[..., :3] * a + 235 * (1 - a)).astype(np.uint8)
    return car, full, model, draw_debug(base, model)


def on_generate(car, full, model, bg, opacity, fade, blur, mode, shine, what="Whole car"):
    if car is None or model is None:
        return None
    if not model.get("ok"):
        raise gr.Error("Could not find a car in this image.")
    h, w = car.shape[:2]
    if bg is None:
        backdrop = np.full((h, w, 3), 235, np.uint8)
    else:
        backdrop = cv2.resize(bg[..., :3], (w, h), interpolation=cv2.INTER_AREA)
    src = wheels_only(car, model) if what == "Wheels only" else car
    rgb, a = make_reflection(src, model, opacity, fade, blur)
    # the reflection goes on the floor first; the shadow and the car are drawn over it
    return composite(backdrop, rgb, a, full, mode, shine)


with gr.Blocks(title="Car Reflection") as demo:
    gr.Markdown(
        "## Automatic perspective-matched car reflection\n"
        "Upload a car (transparent PNG, or any photo: it is cut out automatically) and a backdrop. "
        "Leave room below the car in the image for the reflection.\n\n"
        "Preview: the green line is the floor line the car is mirrored about; red dots are the tyre contacts."
    )
    car_s, full_s, model_s = gr.State(), gr.State(), gr.State()
    with gr.Row():
        with gr.Column():
            inp = gr.Image(label="Car image", type="numpy", image_mode="RGBA")
            bg = gr.Image(label="Backdrop", type="numpy", image_mode="RGB")
            has_shadow = gr.Checkbox(value=True, label="Car image already has a shadow (ignore it when finding the tyres)")
            what = gr.Radio(["Whole car", "Wheels only"], value="Whole car", label="What to reflect")
            opacity = gr.Slider(0, 1, 0.45, step=0.01, label="Reflection opacity")
            fade = gr.Slider(0.05, 1.5, 0.6, step=0.01, label="Fade length (fraction of car height)")
            blur = gr.Slider(0, 30, 3, step=0.5, label="Blur (grows with distance from the car)")
            mode = gr.Dropdown(BLEND_MODES, value="Normal", label="Blend mode")
            shine = gr.Slider(0, 1, 0.0, step=0.01, label="Shine (glossy highlights)")
            go = gr.Button("Generate", variant="primary")
        with gr.Column():
            picker = gr.Image(label="Detected floor lines", interactive=False, type="numpy")
            out = gr.Image(label="Result", format="png")

    gen_inputs = [car_s, full_s, model_s, bg, opacity, fade, blur, mode, shine, what]
    up_out = [car_s, full_s, model_s, picker]
    inp.change(on_upload, [inp, has_shadow], up_out).then(on_generate, gen_inputs, out)
    has_shadow.change(on_upload, [inp, has_shadow], up_out).then(on_generate, gen_inputs, out)
    bg.change(on_generate, gen_inputs, out)
    go.click(on_generate, gen_inputs, out)
    for s in (opacity, fade, blur, shine):
        s.release(on_generate, gen_inputs, out)
    mode.change(on_generate, gen_inputs, out)
    what.change(on_generate, gen_inputs, out)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b0f727aac89f129ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
